# 策略概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

**HDBSCAN PCA-Loadings ResidFDR** 在殘差化因子載荷聚類的基礎上，對群內共整合篩選加入兩道統計嚴謹性強化：

1. 報酬**因子殘差化**（移除市場＋產業因子）→ 5 維 PCA 因子載荷 → HDBSCAN 聚類
2. 群內多道品質過濾（相關初篩、ADF、半衰期、Hurst、Beta 差、波動比、ADV）+ 品質評分排序
3. **BH-FDR 多重檢定校正**：對全部候選配對的共整合 p 值做 Benjamini–Hochberg 校正，控制偽發現率
4. **成本可行性過濾**：剔除 spread 振幅不足以覆蓋往返交易成本（0.58%）的配對
5. 依品質評分取前 `top_n`



## 動機：控制大規模配對搜尋的偽陽性與費用侵蝕

S&P 500 全市場約 $N(N-1)/2 \approx$ 12.5 萬個候選配對。若對每對用固定 $p < 0.05$ 檢定共整合，
純機率下就會產生**數千個偽共整合**——這些配對出樣本不回歸，變成強制平倉的虧損來源。

同時，即使配對真的共整合，若其 spread 振幅太小，往返交易成本（0.58%）會把訊號吃光。
本策略以兩道統計/經濟過濾（多重檢定校正、成本可行性）針對性地解決這兩個問題。


# 參考文獻與引用對應


## 文獻 1：Avellaneda & Lee (2010)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

**參考部分**：報酬因子分解（因子暴露 + 特殊性報酬），統計套利建於殘差報酬。

**為何參考**：階段 1 的因子殘差化與 5 維因子載荷聚類座標之依據（同 Resid 版）。



## 文獻 2：Benjamini & Hochberg (1995)

> Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: A practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B*, **57**(1), 289–300.

**參考部分**：

- **偽發現率（FDR）**控制框架：在 $m$ 個同時檢定中，控制「被拒絕的虛無假設中實為真」的期望比例
- BH 程序：將 p 值升序排列，取最大的 $p_{(k)}$ 使 $p_{(k)} \le \frac{k}{m}\alpha$，作為拒絕門檻

**為何參考**：

- 階段 3 對約 12.5 萬對候選的共整合 p 值直接套用 BH 程序（`_bh_fdr_threshold`）：
  以寬鬆 ADF 門檻（$p<0.10$）放行候選，最後統一以 BH 臨界值過濾——
  比固定 $p<0.01$ 更能同時兼顧「不漏真配對」與「壓制偽共整合」



## 文獻 3：Do & Faff (2012)

> Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research*, **35**(2), 261–287.

**參考部分**：配對交易獲利對往返交易成本高度敏感，低振幅配對在計入成本後多為淨虧損。

**為何參考**：

- 階段 4 成本可行性過濾的依據：要求單次往返的預期擷取 $entry_z \times \sigma_{spread}$
  須 $\ge$ 往返成本（$2 \times 0.29\% = 0.58\%$），在形成期就剔除註定被費用吃光的配對



# 各階段行為


## 階段 1：因子殘差化 + PCA5 因子載荷 + HDBSCAN 聚類

報酬先做因子殘差化（移除市場因子 → 產業因子，見 Resid 版階段 1 公式），
以殘差報酬的 5 維 PCA 因子載荷做 HDBSCAN 聚類（`min_cluster_size=5`、`reduce_method="none"`），噪音點排除。


## 階段 2：群內多濾網品質篩選

群內每對股票依序通過（任一未過即淘汰）：

| 步驟 | 過濾 | 門檻 |
| :---: | :--- | :--- |
| 1 | 相關係數初篩 | $\text{corr}(\ln P_A, \ln P_B) \ge 0.50$ |
| 2 | ADF 共整合（雙向取較小 p） | FDR 模式下用寬鬆閘 $p < 0.10$ 放行（見階段 3） |
| 3 | OU 半衰期 | $1 \le HL \le 63$ 日 |
| 4 | Hurst | $H < 0.55$ |
| 5 | **成本可行性** | $entry_z \times \sigma_{spread} \ge 0.58\%$（見階段 4） |
| 6 | Beta 差異 | $|\beta_A - \beta_B| \le 0.8$ |
| 7 | 波動率比 | $\max/\min \le 3.0$ |
| 8 | ADV 流動性比 | $\ge 0.1$ |
| 9 | 品質評分 | 綜合配對特徵計算 `quality_score`（排序用） |


## 階段 3：BH-FDR 多重檢定校正（依據：Benjamini & Hochberg 1995）

ADF 階段先用**寬鬆閘** $p < 0.10$ 放行候選（避免過早剔除真配對），
記錄做過 ADF 檢定的配對總數 $m$。全部候選收集完後，統一套用 BH 程序：

$$p_{crit} = \max\left\{ p_{(k)} : p_{(k)} \le \frac{k}{m}\,\alpha \right\}, \qquad \alpha = 0.05$$

只保留 $\text{ADF\_PValue} \le p_{crit}$ 的配對。$m$ 以全部做過 ADF 的配對數計入
（未進 records 的以 $p=1$ 補齊），確保多重檢定的分母正確。


## 階段 4：成本可行性過濾（依據：Do & Faff 2012）

一次往返（$entry_z\cdot\sigma$ 進場 → 回到 0 出場）的預期分數擷取 $\approx entry_z \times \sigma_{spread}$，
須顯著大於往返成本：

$$entry_z \times \sigma_{spread} \ \ge\ \text{margin} \times \text{roundtrip\_cost}, \qquad
entry_z = 2.0,\ \text{roundtrip\_cost} = 0.0058,\ \text{margin} = 1.0$$

$\sigma_{spread}$ 為 log-price 共整合殘差的標準差（近似分數移動幅度）。不滿足者於形成期即剔除。


## 階段 5：排序與交易期銜接

通過全部過濾與 BH-FDR 校正的配對依 `quality_score` 取前 `top_n`。
輸出 `OLS_Alpha`（本策略共整合殘差於 log-price 空間），交易期依 config 的
`ignore_ols_alpha` 設定決定 spread 重建座標。形成期統計量整個交易期凍結（無前視）。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗 $F$ / 滾動步長 | 252 / 21 交易日 | — |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 每期配對數 |
| `factor_residual` | **True** | 移除市場＋產業因子 |
| `feature_mode` | `pca_loadings` | 5 維殘差因子載荷 |
| `use_fdr` / `fdr_alpha` | **True** / 0.05 | BH-FDR 多重檢定校正 |
| `use_cost_filter` / `roundtrip_cost` / `cost_margin` | **True** / 0.0058 / 1.0 | 成本可行性 |
| `adf_pvalue_threshold` | 0.01（FDR 模式閘放寬至 0.10） | 共整合 |
| `min_corr` / `halflife_max` / `hurst_threshold` | 0.50 / 63 / 0.55 | 品質過濾 |
| `max_beta_diff` / `max_vol_ratio` / `min_adv_ratio` | 0.8 / 3.0 / 0.1 | 品質過濾 |
